
### Notebook to reformat the structure of .cmf files

This Notebook is deliberately very simple as it needs modified for each new object since there isn't a 'default' directory structure for the bundles of .cmf files fetched from the stamp server.



In [ ]:

""" importing packages """

from glob import glob

from astropy.io import fits


In [ ]:

""" user-defined things """

# [OPTIONAL] Define some date to truncate the list of .cmfs.
# Only necessary if the directory contains lots of historical data
MJD_start = 50000 # default (some really old MJD value): 50000

# defining the path to the CMF files
CMF_filepathnames = sorted(glob("../cmf_files/*.cmf"))

print(f"Number of .cmf files to process: \t {len(CMF_filepathnames)}")

CMF_filepathnames


In [ ]:

def read_metadata():
    """ useful for reading in transient metadata """
    
    data = {}

    with open("metadata.txt", "r") as f:
        for line in f:
            line = line.strip()

            # Skip empty lines and headers
            if not line or line.startswith("#"):
                continue

            key, value = line.split(":", 1)

            data[key.strip()] = value.strip()

    transient_name = data["transient_name"]
    ra = data["RA"]
    dec = data["Dec"]

    print("transient_name, ra, dec")
    print(transient_name, ra, dec)
    
    return transient_name, ra, dec


In [ ]:

""" running some code to copy the .cmf files into a new, structured directory """

transient_name, ra, dec = read_metadata()

counter = 1

folder_name = "CMFs"

# only need to define the new folder name once, so placing this command here
cmd = f"mkdir ../Bundles/{folder_name}"
! {cmd}

for CMF_filepathname in CMF_filepathnames:

    # load in the file
    hdul = fits.open(CMF_filepathname)
    # print(hdul.info())

    # contains the table of sources and their photometry (+ other info)
    data = hdul[1].data

    # contains useful metadata for the stack (e.g., exp. time, filter)
    header = hdul[0].header

    #############################################################################   
    #############################################################################   


    # extract MJD from header
    MJD_entries = {k: v for k, v in header.items() if k.startswith("MJD-OBS")}

    MJD = int(list(MJD_entries.values())[0])

    print(CMF_filepathname, "\t\t", MJD)

    if MJD > MJD_start:

        filepath = f"../Bundles/{folder_name}/MJD{MJD}_Exposure-{counter:03d}"

        # need to make a new exposure_N folder for each exposure
        cmd = f"mkdir {filepath}"
        ! {cmd}

        # and now we are copying the file across to the new directory
        cmd = f"cp {CMF_filepathname} {filepath}"
        ! {cmd}

        counter += 1
